# Sweeping an input value

A parameter sweep (notebook 02) changes a *tuned constant* the pipeline reads
via `param()`. An input sweep changes a *record's own value* instead —
`income` here, a column in the input frame, not a parameter. `overrides`
without an `at` writes it before anything runs, because it's an input, not a
value the pipeline produced.

In [1]:
import polars as pl
from decider.experiments import Experiment, OutcomeCounts, RunStatus
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [350.0, 900.0, 700.0, 300.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 350.0 │
│ 2000.0 ┆ 900.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 300.0 │
└────────┴───────┘


In [2]:
exp = Experiment("income-sweep", pipeline, df, entry=ENTRY)
for income in (600.0, 900.0, 1200.0, 3000.0):
    exp.scenario(f"income_{income}", overrides={"income": income})
exp.plot(OutcomeCounts(type="outcome_counts", column="approved")).compare(outputs=("approved",))

result = exp.run()
assert result.status is RunStatus.COMPLETED
[(s.name, s.status.value, len(s.divergences)) for s in result.scenarios]

[('income_600.0', 'completed', 2), ('income_900.0', 'completed', 0), ('income_1200.0', 'completed', 0), ('income_3000.0', 'completed', 2)]


`validate_scenarios` is what makes this safe to hand to a client: overriding
a *produced* value needs a declared `at` (the step it overrides), but an
*input* column like `income` needs nothing extra, because it's settable
before anything runs.

In [3]:
from decider.experiments import validate_scenarios
validate_scenarios(exp.def_, pipeline)

[]
